In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

# Students performance in exams
#### Marks secured by the students in college

## Aim
#### To understand the influence of various factors like economic, personal and social on the students performance 

## Inferences would be : 
#### 1. How to imporve the students performance in each test ?
#### 2. What are the major factors influencing the test scores ?
#### 3. Effectiveness of test preparation course?
#### 4. Other inferences 

#### Import the required libraries

In [ ]:
%%RecordEventWithColumnInfo
import numpy as np
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time
import cudf as cd 

#### Let us initialize the required values ( we will use them later in the program )
#### we will set the minimum marks to 40 to pass in a exam

In [ ]:
%%RecordEventWithColumnInfo
passmark = 40

#### Let us read the data from the csv file

In [ ]:
%%RecordEventWithColumnInfo
### cell 0 ###

df = pd.read_csv("/home/dias-benchmarks/notebooks/spscientist/student-performance-in-exams/input/StudentsPerformance.csv")

# -- STEFANOS -- Replicate Data

In [ ]:
%%RecordEventWithColumnInfo
### cell 1 ###

factor = 10
df = pd.concat([df]*factor)
df.info()

#### We will print top few rows to understand about the various data columns

#### Size of data frame

In [ ]:
%%RecordEventWithColumnInfo
### cell 2 ###

df.head()

In [ ]:
%%RecordEventWithColumnInfo
### cell 3 ###

print(df.shape)

#### Let us understand about the basic information of the data, like min, max, mean and standard deviation etc.

In [ ]:
%%RecordEventWithColumnInfo
### cell 4 ###

df.describe()

In [10]:
# ##move the dataframe to GPU
# df = cd.DataFrame.from_pandas(df)

#### Let us check for any missing values

In [ ]:
%%RecordEventWithColumnInfo
### cell 5 ###

df.isnull().sum()

##### As seen above, there are no missing ( null ) values in this dataframe but in real scenarios we need work on dataset with a lot of missing values  

####  Let us explore the Math Score first

In [12]:
# STEFANOS: Disable plotting
# p = sns.countplot(x="math score", data = df, palette="muted")
# _ = plt.setp(p.get_xticklabels(), rotation=90) 

#### How many students passed in Math exam ?

In [13]:
# %%timeit
# df['math score'] = pd.to_numeric(df['math score'], errors='coerce')
# df['Math_PassStatus'] = np.where(df['math score']<passmark, 'F', 'P')
# df.Math_PassStatus.value_counts()

In [ ]:
%%RecordEventWithColumnInfo
### cell 6 ###

df['math score'] = df['math score'].astype(float).fillna(-1)  # Replace NaNs with a default value

# Vectorized conditional assignment using cuDF
df['Math_PassStatus'] = (df['math score'] >= passmark).astype("str").replace({'True': 'P', 'False': 'F'})

# Use cuDF's optimized value_counts()
df['Math_PassStatus'].value_counts()

In [15]:
# STEFANOS: Disable plotting
# p = sns.countplot(x='parental level of education', data = df, hue='Math_PassStatus', palette='bright')
# _ = plt.setp(p.get_xticklabels(), rotation=90) 

#### Let us explore the Reading score

In [16]:
# STEFANOS: Disable plotting
# sns.countplot(x="reading score", data = df, palette="muted")
# plt.show()

#### How many studends passed in reading ?

In [17]:

# df['reading score'] = pd.to_numeric(df['reading score'], errors='coerce')
# df['Reading_PassStatus'] = np.where(df['reading score']<passmark, 'F', 'P')
# df.Reading_PassStatus.value_counts()

In [ ]:
%%RecordEventWithColumnInfo
### cell 7 ###


#rewritten
# Convert 'reading score' to numeric (cuDF does not have errors='coerce', so use fillna after)
df['reading score'] = df['reading score'].astype(float).fillna(-1)  # Replace NaNs if needed

# Use cuDF's vectorized conditional assignment
df['Reading_PassStatus'] = (df['reading score'] >= passmark).astype("str").replace({'True': 'P', 'False': 'F'})

# Use cuDF's value_counts() (faster than pandas on GPU)
df['Reading_PassStatus'].value_counts()


In [19]:
# STEFANOS: Disable plotting
# p = sns.countplot(x='parental level of education', data = df, hue='Reading_PassStatus', palette='bright')
# _ = plt.setp(p.get_xticklabels(), rotation=90) 

#### Let us explore writing score

In [20]:
# STEFANOS: Disable plotting
# p = sns.countplot(x="writing score", data = df, palette="muted")
# _ = plt.setp(p.get_xticklabels(), rotation=90) 

#### How many students passed writing ?

In [21]:

# df['writing score'] = pd.to_numeric(df['writing score'], errors='coerce')
# df['Writing_PassStatus'] = np.where(df['writing score']<passmark, 'F', 'P')
# df.Writing_PassStatus.value_counts()

In [ ]:
%%RecordEventWithColumnInfo
### cell 8 ###


#rewritten
# Convert 'reading score' to numeric (cuDF does not have errors='coerce', so use fillna after)
df['reading score'] = df['reading score'].astype(float).fillna(-1)  # Replace NaNs if needed

# Use cuDF's vectorized conditional assignment
df['Reading_PassStatus'] = (df['reading score'] >= passmark).astype("str").replace({'True': 'P', 'False': 'F'})

# Use cuDF's value_counts() (faster than pandas on GPU)
df['Reading_PassStatus'].value_counts()

##rewritten
df['writing score'] = df['writing score'].astype('float32')

# Use cuDF's boolean indexing + direct assignment (avoids unnecessary operations)
df['Writing_PassStatus'] = 'F'  # Default all to 'F' first
df.loc[df['writing score'] >= passmark, 'Writing_PassStatus'] = 'P'  # Assign 'P' where needed

# Efficient value_counts() with cuDF
df['Writing_PassStatus'].value_counts()

In [23]:
# STEFANOS: Disable plotting
# p = sns.countplot(x='parental level of education', data = df, hue='Writing_PassStatus', palette='bright')
# _ = plt.setp(p.get_xticklabels(), rotation=90) 

#### Iet us check "How many students passed in all the subjects ?"

In [24]:
# %%timeit
# df['OverAll_PassStatus'] = df.apply(lambda x : 'F' if x['Math_PassStatus'] == 'F' or 
#                                     x['Reading_PassStatus'] == 'F' or x['Writing_PassStatus'] == 'F' else 'P', axis =1)

# df.OverAll_PassStatus.value_counts()

In [ ]:
%%RecordEventWithColumnInfo
### cell 9 ###

df['OverAll_PassStatus'] = ((df['Math_PassStatus'] == 'P') & 
                            (df['Reading_PassStatus'] == 'P') & 
                            (df['Writing_PassStatus'] == 'P')).astype("str").replace({'True': 'P', 'False': 'F'})

# Optimized cuDF value_counts()
df['OverAll_PassStatus'].value_counts()

In [26]:

# # Assuming 'df_pandas' is your original pandas DataFrame
# # Convert to cuDF DataFrame if it's not already


# # Optimization for cuDF - Vectorized operations instead of apply

# # 1. Create boolean Series for each PassStatus column being 'F'
# math_fail = df['Math_PassStatus'] == 'F'
# reading_fail = df['Reading_PassStatus'] == 'F'
# writing_fail = df['Writing_PassStatus'] == 'F'

# # 2. Combine the boolean Series using logical OR (|)
# # This creates a boolean Series where True indicates at least one Fail status
# overall_fail_condition = math_fail | reading_fail | writing_fail

# # 3. Use cudf.Series.where to create 'OverAll_PassStatus' based on the combined condition
# #  - where(condition, other=...) : If condition is True, keep original (or replace with specified 'other' for True case if provided, otherwise NaN),
# #                                   if condition is False, replace with 'other'.
# #  - In this case, condition is 'overall_fail_condition'.
# #  - We want 'F' if overall_fail_condition is True (at least one fail), and 'P' otherwise.
# df['OverAll_PassStatus'] = df['Math_PassStatus'].where(overall_fail_condition, 'P').fillna('F')
# # Explanation of .where and .fillna:
# #   - .where(overall_fail_condition, 'P') : Where overall_fail_condition is TRUE (meaning at least one subject failed), keep the original 'Math_PassStatus' value (which is irrelevant here, just a placeholder). Where overall_fail_condition is FALSE (all subjects passed), replace with 'P'.
# #   - .fillna('F') : Then, fill the remaining NaN values (which are where overall_fail_condition was TRUE and we kept the original 'Math_PassStatus' value) with 'F'.
# #     This effectively assigns 'F' when any subject fails and 'P' when all subjects pass.

# # Alternative and potentially more efficient approach using .mask (might be slightly clearer in intent):
# # Initialize with 'P' and then mask where overall_fail_condition is True to 'F'
# # df_cudf['OverAll_PassStatus'] = 'P'  # Initialize all to 'P' (Pass)
# # df_cudf['OverAll_PassStatus'] = df_cudf['OverAll_PassStatus'].mask(overall_fail_condition, 'F')
# # Explanation of .mask:
# #   - .mask(condition, other): Where condition is TRUE, replace with 'other'.
# #   - Here, condition is 'overall_fail_condition'.
# #   - So, where overall_fail_condition is TRUE (at least one fail), replace 'OverAll_PassStatus' with 'F'. Otherwise, it remains 'P' (as initialized).


# # 4. Get value counts using cuDF's value_counts
# overall_pass_status_counts_cudf = df['OverAll_PassStatus'].value_counts()

# # If you need to convert the results back to pandas for further processing (if needed):


#  # Or print(overall_pass_status_counts_cudf)

In [27]:
# STEFANOS: Disable plotting
# p = sns.countplot(x='parental level of education', data = df, hue='OverAll_PassStatus', palette='bright')
# _ = plt.setp(p.get_xticklabels(), rotation=90) 

#### Find the percentage of marks

In [ ]:
%%RecordEventWithColumnInfo
### cell 10 ###

df['Total_Marks'] = df['math score']+df['reading score']+df['writing score']
df['Percentage'] = df['Total_Marks']/3

In [29]:
# STEFANOS: Disable plotting
# p = sns.countplot(x="Percentage", data = df, palette="muted")
# _ = plt.setp(p.get_xticklabels(), rotation=0) 

#### Let us assign the grades

### Grading 
####    above 80 = A Grade
####      70 to 80 = B Grade
####      60 to 70 = C Grade
####      50 to 60 = D Grade
####      40 to 50 = E Grade
####    below 40 = F Grade  ( means Fail )


In [30]:
# %%timeit
# def GetGrade(Percentage, OverAll_PassStatus):
#     if ( OverAll_PassStatus == 'F'):
#         return 'F'    
#     if ( Percentage >= 80 ):
#         return 'A'
#     if ( Percentage >= 70):
#         return 'B'
#     if ( Percentage >= 60):
#         return 'C'
#     if ( Percentage >= 50):
#         return 'D'
#     if ( Percentage >= 40):
#         return 'E'
#     else: 
#         return 'F'

# df['Grade'] = df.apply(lambda x : GetGrade(x['Percentage'], x['OverAll_PassStatus']), axis=1)

# df.Grade.value_counts()

In [ ]:
%%RecordEventWithColumnInfo
### cell 11 ###


##rewritten
# Initialize Grade column with 'F' (default for failures)
df['Grade'] = 'F'

# Apply vectorized conditional assignments
df.loc[df['OverAll_PassStatus'] == 'F', 'Grade'] = 'F'
df.loc[(df['Percentage'] >= 80) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'A'
df.loc[(df['Percentage'] >= 70) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'B'
df.loc[(df['Percentage'] >= 60) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'C'
df.loc[(df['Percentage'] >= 50) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'D'
df.loc[(df['Percentage'] >= 40) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'E'

# Efficient value counts
df['Grade'].value_counts()


#### we will plot the grades obtained in a order

In [32]:
# STEFANOS: Disable plotting
# sns.countplot(x="Grade", data = df, order=['A','B','C','D','E','F'],  palette="muted")
# plt.show()

In [33]:
# STEFANOS: Disable plotting
# p = sns.countplot(x='parental level of education', data = df, hue='Grade', palette='bright')
# _ = plt.setp(p.get_xticklabels(), rotation=90) 